# Notebook 01 — Prime chiamate all'API di un LLM

**Settimana:** 1
**Blocco:** 3.1 — SDK e API
**Tempo stimato:** 2-3 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Autenticazione sicura con OpenAI e Anthropic
- Una funzione riutilizzabile per chiamare uno qualsiasi dei due provider
- Un esperimento sull'effetto di `temperature` visto con i tuoi occhi
- Una funzione che stima il costo di una chiamata prima di effettuarla
- Un chatbot da console funzionante con cronologia

## Prerequisiti

```bash
pip install openai anthropic python-dotenv tiktoken
```

Hai bisogno di un file `.env` nella root del progetto con:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

---
## Cella 0 — Setup: installazione e caricamento delle chiavi

Esegui sempre questa cella all'inizio. Verifica che le chiavi vengano caricate correttamente.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

# Verifica sicura: mostra solo i primi 8 caratteri
def verificar_clave(nombre: str, clave: str | None) -> None:
    if clave:
        print(f"✓ {nombre}: {clave[:8]}...")
    else:
        print(f"✗ {nombre}: NON trovata. Controlla il tuo .env")

verificar_clave("OPENAI_API_KEY", OPENAI_KEY)
verificar_clave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)

---
## Cella 1 — Prima chiamata a OpenAI

La chiamata più semplice possibile. Osserva:
- Come viene inizializzato il client
- La struttura del parametro `messages`
- Come estrarre il testo dalla risposta
- Quali informazioni restituisce `response.usage`

In [ ]:
from openai import OpenAI

cliente_openai = OpenAI(api_key=OPENAI_KEY)

respuesta_openai = cliente_openai.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[
        {"role": "system", "content": "Sei un assistente conciso. Rispondi sempre in una frase."},
        {"role": "user", "content": "Cos'è un token nel contesto degli LLM?"}
    ]
)

# Il testo della risposta
texto = respuesta_openai.choices[0].message.content
print("Risposta:", texto)

# Informazioni di utilizzo
uso = respuesta_openai.usage
print(f"\nToken di input:  {uso.prompt_tokens}")
print(f"Token di output: {uso.completion_tokens}")
print(f"Totale:           {uso.total_tokens}")

---
## Cella 2 — Prima chiamata ad Anthropic

La stessa domanda, con l'SDK di Anthropic. Nota le differenze:
- `system` è un parametro separato, non va dentro `messages`
- La risposta si trova in `response.content[0].text`
- I token si chiamano `input_tokens` e `output_tokens`

In [ ]:
import anthropic

cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

respuesta_anthropic = cliente_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    system="Sei un assistente conciso. Rispondi sempre in una frase.",
    messages=[
        {"role": "user", "content": "Cos'è un token nel contesto degli LLM?"}
    ]
)

# Il testo della risposta
texto = respuesta_anthropic.content[0].text
print("Risposta:", texto)

# Informazioni di utilizzo
uso = respuesta_anthropic.usage
print(f"\nToken di input:  {uso.input_tokens}")
print(f"Token di output: {uso.output_tokens}")

---
## Cella 3 — Esperimento: l'effetto di `temperature`

Lancia la stessa domanda creativa con 5 valori diversi di `temperature`.
Osserva come varia la risposta e trai le tue conclusioni.

**Prima di eseguire:** cosa ti aspetti che succeda con `temperature=0`? E con `temperature=1.5`?

In [ ]:
PREGUNTA_CREATIVA = "Inventa un nome per una startup di IA che aiuta gli agricoltori."
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]

print(f"Domanda: {PREGUNTA_CREATIVA}\n")
print("=" * 60)

for temp in TEMPERATURES:
    respuesta = cliente_openai.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=50,
        temperature=temp,
        messages=[
            {"role": "user", "content": PREGUNTA_CREATIVA}
        ]
    )
    texto = respuesta.choices[0].message.content.strip()
    print(f"temperature={temp:.1f} → {texto}")

print("=" * 60)
print("\nCosa osservi? Scrivi le tue conclusioni nella cella successiva.")

**Le tue osservazioni** (modifica questa cella):

- Con `temperature=0.0`: ...
- Con `temperature=1.5`: ...
- Il punto di equilibrio per questo caso d'uso mi sembra: ...
- Per l'estrazione di dati userei temperature=... perché ...

---
## Cella 4 — Funzione per stimare il costo

Prima di chiamare l'API in produzione, conviene sempre sapere quanto costerà.
Costruisci una funzione riutilizzabile da applicare in tutti i notebook successivi.

In [ ]:
# Prezzi in USD per milione di token (aggiorna se cambiano)
PRECIOS = {
    "gpt-4o-mini": {
        "input":  0.150,
        "output": 0.600
    },
    "gpt-4o": {
        "input":  2.50,
        "output": 10.00
    },
    "claude-haiku-4-5-20251001": {
        "input":  0.080,
        "output": 0.400
    },
    "claude-sonnet-4-6": {
        "input":  3.00,
        "output": 15.00
    }
}

def calcular_coste(
    tokens_input: int,
    tokens_output: int,
    modelo: str
) -> dict:
    """Calcola il costo in USD di una chiamata."""
    if modelo not in PRECIOS:
        raise ValueError(f"Modello '{modelo}' non trovato nella tabella dei prezzi.")

    precios = PRECIOS[modelo]
    coste_input  = (tokens_input  / 1_000_000) * precios["input"]
    coste_output = (tokens_output / 1_000_000) * precios["output"]
    coste_total  = coste_input + coste_output

    return {
        "modelo":        modelo,
        "tokens_input":  tokens_input,
        "tokens_output": tokens_output,
        "coste_input_usd":  round(coste_input,  6),
        "coste_output_usd": round(coste_output, 6),
        "coste_total_usd":  round(coste_total,  6),
        "coste_total_eur":  round(coste_total * 0.92, 6)  # tasso approssimativo
    }

def proyectar_coste_mensual(
    tokens_input_por_query: int,
    tokens_output_por_query: int,
    modelo: str,
    queries_por_dia: int,
    usuarios: int
) -> None:
    """Proietta il costo mensile di un'applicazione."""
    queries_totales_dia = queries_por_dia * usuarios
    coste_por_query = calcular_coste(tokens_input_por_query, tokens_output_por_query, modelo)
    coste_dia   = coste_por_query["coste_total_usd"] * queries_totales_dia
    coste_mes   = coste_dia * 30

    print(f"Modello:                 {modelo}")
    print(f"Utenti:                  {usuarios:,}")
    print(f"Query/utente/giorno:     {queries_por_dia}")
    print(f"Query totali/giorno:     {queries_totales_dia:,}")
    print(f"Costo per query:         ${coste_por_query['coste_total_usd']:.6f}")
    print(f"Costo giornaliero:       ${coste_dia:.2f}")
    print(f"Costo mensile stimato:   ${coste_mes:.2f} / €{coste_mes * 0.92:.2f}")


# --- Test 1: costo della chiamata dell'esperimento precedente ---
print("=== Costo di una singola chiamata ===")
resultado = calcular_coste(
    tokens_input=150,
    tokens_output=50,
    modelo="gpt-4o-mini"
)
for k, v in resultado.items():
    print(f"  {k}: {v}")

print()

# --- Test 2: proiezione mensile ---
print("=== Proiezione mensile: 1.000 utenti, 5 query/giorno ===")
proyectar_coste_mensual(
    tokens_input_por_query=300,
    tokens_output_por_query=150,
    modelo="gpt-4o-mini",
    queries_por_dia=5,
    usuarios=1000
)

**Esercizio:** usa `proyectar_coste_mensual` per confrontare `gpt-4o-mini` con `claude-haiku` usando gli stessi parametri. Quale modello è più economico?

In [ ]:
# La tua comparativa qui
# Suggerimento: chiama proyectar_coste_mensual due volte, una per modello

print("=== gpt-4o-mini ===")
# ...

print("\n=== claude-haiku ===")
# ...

---
## Cella 5 — Wrapper generico multi-provider

Il pattern più utile che tu possa avere: una funzione che accetta il provider come parametro
e astrae le differenze tra gli SDK. Usala come base in tutti i tuoi progetti.

In [ ]:
from typing import Literal

# Modelli predefiniti per ciascun provider
MODELOS_DEFAULT = {
    "openai":    "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001"
}

def llamar_llm(
    mensaje_usuario: str,
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system_prompt: str = "Sei un assistente utile e conciso.",
    modelo: str | None = None,
    temperature: float = 0.7,
    max_tokens: int = 512
) -> dict:
    """
    Wrapper unificato per chiamare OpenAI o Anthropic.

    Restituisce un dict con:
      - texto: la risposta del modello
      - tokens_input: token consumati in input
      - tokens_output: token consumati in output
      - modelo: modello usato
      - proveedor: provider usato
    """
    modelo = modelo or MODELOS_DEFAULT[proveedor]

    if proveedor == "openai":
        respuesta = cliente_openai.chat.completions.create(
            model=modelo,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": mensaje_usuario}
            ]
        )
        return {
            "texto":         respuesta.choices[0].message.content,
            "tokens_input":  respuesta.usage.prompt_tokens,
            "tokens_output": respuesta.usage.completion_tokens,
            "modelo":        modelo,
            "proveedor":     proveedor
        }

    elif proveedor == "anthropic":
        respuesta = cliente_anthropic.messages.create(
            model=modelo,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[
                {"role": "user", "content": mensaje_usuario}
            ]
        )
        return {
            "texto":         respuesta.content[0].text,
            "tokens_input":  respuesta.usage.input_tokens,
            "tokens_output": respuesta.usage.output_tokens,
            "modelo":        modelo,
            "proveedor":     proveedor
        }

    else:
        raise ValueError(f"Provider '{proveedor}' non supportato. Usa 'openai' o 'anthropic'.")


# --- Test del wrapper ---
PREGUNTA = "Spiega cos'è un embedding in una frase."

for prov in ["openai", "anthropic"]:
    resultado = llamar_llm(PREGUNTA, proveedor=prov)
    coste = calcular_coste(resultado["tokens_input"], resultado["tokens_output"], resultado["modelo"])
    print(f"[{prov.upper()} / {resultado['modelo']}]")
    print(f"  Risposta: {resultado['texto']}")
    print(f"  Token:    {resultado['tokens_input']} input + {resultado['tokens_output']} output")
    print(f"  Costo:    ${coste['coste_total_usd']:.6f}")
    print()

---
## Sfida finale — Chatbot da console con cronologia

Costruisci un chatbot interattivo che:
- Mantenga la cronologia della conversazione tra i turni
- Mostri il costo accumulato ad ogni turno
- Permetta di scegliere il provider all'avvio
- Si chiuda in modo pulito con `esci` o `exit`

> **Nota:** questa sfida usa `input()`, che in alcuni ambienti non funziona direttamente in Jupyter.
> Se hai problemi, copia il codice in un file `chatbot.py` ed eseguilo dal terminale.

Leggi il codice, comprendilo, e **completa i frammenti contrassegnati con `# TODO`**.

In [ ]:
def chatbot_consola(
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system_prompt: str = "Sei un assistente utile e conciso. Rispondi sempre in italiano."
) -> None:
    """
    Chatbot interattivo con cronologia della conversazione.
    La cronologia si accumula in una lista di messaggi che cresce ad ogni turno.
    """
    historial: list[dict] = []  # lista di {role, content}
    coste_acumulado = 0.0
    modelo = MODELOS_DEFAULT[proveedor]

    print(f"\nChatbot avviato | Provider: {proveedor} | Modello: {modelo}")
    print("Scrivi 'esci' per terminare.\n")

    while True:
        # 1. Leggere l'input dell'utente
        mensaje_usuario = input("Tu: ").strip()
        if not mensaje_usuario:
            continue
        if mensaje_usuario.lower() in ("esci", "exit", "quit"):
            print(f"\nCosto totale della sessione: ${coste_acumulado:.6f} USD")
            print("A presto!")
            break

        # 2. Aggiungere il messaggio dell'utente alla cronologia
        historial.append({"role": "user", "content": mensaje_usuario})

        # 3. Chiamare l'LLM con la cronologia completa
        # TODO: implementa la chiamata. A differenza di llamar_llm(), qui
        # devi passare tutta la cronologia, non solo l'ultimo messaggio.
        # Suggerimento: devi chiamare direttamente cliente_openai o cliente_anthropic
        # con messages=historial (più il system prompt se è Anthropic).

        if proveedor == "openai":
            mensajes_con_system = [
                {"role": "system", "content": system_prompt}
            ] + historial
            respuesta = cliente_openai.chat.completions.create(
                model=modelo,
                max_tokens=512,
                messages=mensajes_con_system
            )
            texto_respuesta  = respuesta.choices[0].message.content
            tokens_input     = respuesta.usage.prompt_tokens
            tokens_output    = respuesta.usage.completion_tokens

        else:  # anthropic
            respuesta = cliente_anthropic.messages.create(
                model=modelo,
                max_tokens=512,
                system=system_prompt,
                messages=historial
            )
            texto_respuesta  = respuesta.content[0].text
            tokens_input     = respuesta.usage.input_tokens
            tokens_output    = respuesta.usage.output_tokens

        # 4. Aggiungere la risposta dell'assistente alla cronologia
        historial.append({"role": "assistant", "content": texto_respuesta})

        # 5. Calcolare e accumulare il costo
        coste_turno = calcular_coste(tokens_input, tokens_output, modelo)
        coste_acumulado += coste_turno["coste_total_usd"]

        # 6. Mostrare risposta e metriche
        print(f"\nAssistente: {texto_respuesta}")
        print(
            f"[token: {tokens_input}↑ {tokens_output}↓ | "
            f"turno: ${coste_turno['coste_total_usd']:.6f} | "
            f"totale: ${coste_acumulado:.6f}]\n"
        )


# Decommenta per eseguire in Jupyter (oppure copia in chatbot.py ed esegui dal terminale)
# chatbot_consola(proveedor="anthropic")

### Estensioni opzionali della sfida (se finisci in anticipo)

1. **Limite di contesto:** la cronologia cresce senza limiti, il che rende ogni turno più costoso. Implementa una strategia per mantenere solo gli ultimi N messaggi.

2. **Cambio di provider a caldo:** aggiungi un comando speciale `/cambiar openai` o `/cambiar anthropic` che cambi il provider senza riavviare la conversazione.

3. **Salvare la conversazione:** all'uscita, salva la cronologia in un file JSON con il timestamp come nome.

In [ ]:
# Spazio per le estensioni opzionali

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 01 — Settimana 1 del Piano di Formazione AI Engineering*